# Lecture 9. Logit and probit for a yes/no outcome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture09/lecture09.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Wooldridge, *Introductory Econometrics*, sections 17-1a to 17-1d
and Appendix C-4b.

In Lecture 8 a linear probability model (LPM) for churn gave 41 customers a predicted probability below 0. Today we
work through a model whose predictions stay between 0 and 1, step by step:

| Part | What we do |
|---|---|
| 1 | Set up a model for a yes/no outcome: the logistic curve, and what a logit coefficient is |
| 2 | Fit the model and read the output: how the coefficients are found, and comparing two models |
| 3 | Predicted probabilities for customers we describe, and why the membership gap differs between them |
| 4 | Average marginal effects, probit, and the choice between the LPM, logit and probit |

The explanatory variables are the same as in the Lecture 8 LPM, so every result can be put next to it. Where Python
does something differently from Wooldridge, the notebook says so. Each part ends with an exercise: you predict a
result, run the cell and answer a question; in some you also change one value. You do not need to write any code. The last section
holds the steps, the symbols and the Danish terms.

Run the setup cell first. It loads the RetailCorp customer data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 customers drawn at random, as in Lectures 6 to 8
customers["log_spend"] = np.log(customers["spend_2025"])

print(f"Customers: {len(customers):,}")
print(f"Share who churned: {customers['churned'].mean():.3f}")
customers.head()

## The case and the question

The same 1,000 RetailCorp customers as in Lectures 6 to 8. The outcome is `churned`: 1 if the customer placed no
order in January to June 2026, else 0. The explanatory variables are those of the Lecture 8 LPM: `member` (1 if in the
loyalty club on 1 January 2025), `channel` (App or Web, the main channel in 2025), `tenure_years`, `age` and
`visits_2025`.

**Lead question.** Which customers are likely to stop buying, and how much lower is churn among club members?

**Where Lecture 8 left churn.** The LPM below says members were 10.7 percentage points less likely to churn, at the
same channel, tenure, age and visits. But 41 customers get a predicted probability below 0, and the gap for members is
the same 10.7 points for every customer.

In [ ]:
lpm_formula = "churned ~ member + C(channel) + tenure_years + age + visits_2025"
m_lpm = smf.ols(lpm_formula, data=customers).fit(cov_type="HC1")
print(m_lpm.params.round(4))
print(f"Predicted probabilities below 0: {(m_lpm.fittedvalues < 0).sum()}, lowest {m_lpm.fittedvalues.min():.2f}")

## Part 1. A model for a yes/no outcome

A model for churn predicts, for each customer, the probability of churn. A probability lies between 0 and 1, so the
predictions must stay there for every customer we could describe. A straight line does not. We want a curve that is
steep where churn is uncertain and flattens out as it approaches 0 or 1. Two such curves are used in most
applications (Wooldridge 17-1a): the logistic curve (logit) and the normal curve (probit, Part 4).

### The logistic curve

p<sub>i</sub> = 1 / (1 + e<sup>−z<sub>i</sub></sup>), read "p-i equals one over one plus e to the minus z-i".

- z<sub>i</sub> is the **index** for customer i: z<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> +
  β<sub>2</sub> web<sub>i</sub> + β<sub>3</sub> tenure<sub>i</sub> + β<sub>4</sub> age<sub>i</sub> +
  β<sub>5</sub> visits<sub>i</sub>, the same sum as in every regression so far. It can be any number.
- e is about 2.718, as in Lecture 8.
- p<sub>i</sub> is the predicted probability that customer i churns. Whatever z is, p stays between 0 and 1. The
  curve is steepest at z = 0, where p = 0.5, and flat near 0 and 1.

Wooldridge writes the curve as G(z) or Λ(z), read lambda. So a logit model works in two steps: first the index, then
the probability. The coefficients act on z. How much a change in z moves p depends on where on the curve the customer
is, which is why we read sizes as probabilities later.

In [ ]:
for z in [-2, 0, 2]:
    p = 1 / (1 + np.exp(-z))
    print(f"z = {z:2d}: p = {p:.2f}")

z = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 4))
plt.plot(z, 1 / (1 + np.exp(-z)), label="logistic curve")
plt.plot(z, 0.5 + 0.13 * z, linestyle="--", label="a straight line")
plt.axhline(0, color="gray", linewidth=0.8)
plt.axhline(1, color="gray", linewidth=0.8)
plt.xlabel("z, the index")
plt.ylabel("Predicted probability p")
plt.legend()
plt.show()

### Logit with one yes/no variable

`smf.logit` works like `smf.ols`, with the same kind of formula.

In [ ]:
m_one = smf.logit("churned ~ member", data=customers).fit()
print(m_one.params.round(4))

print(customers.groupby("member")["churned"].agg(["sum", "count", "mean"]).round(3))

**By hand.** z<sub>i</sub> = −0.5478 − 0.9837 member<sub>i</sub>.

- Non-members: 1 / (1 + e<sup>0.5478</sup>) = 0.366
- Members: 1 / (1 + e<sup>0.5478 + 0.9837</sup>) = 0.178

The logit on one dummy gives back the two shares, 218 of 595 and 72 of 405, as the regression on one dummy gave back
the two group means in Lecture 8. The coefficient −0.98 is not a difference in probability: the two shares differ by 18.9
percentage points (from the unrounded shares).

In [ ]:
b = m_one.params
p_non = 1 / (1 + np.exp(-b["Intercept"]))
p_mem = 1 / (1 + np.exp(-(b["Intercept"] + b["member"])))
print(f"Non-members: {p_non:.3f}")
print(f"Members:     {p_mem:.3f}")

### Probability, odds and log odds

The odds are p / (1 − p): odds of 0.578 mean about 58 churners for every 100 who stay.

| Way to state it | Non-members | Members | Members against non-members |
|---|---|---|---|
| Probability | 36.6% | 17.8% | 18.9 percentage points lower |
| Relative to non-members | | | 51% lower probability |
| Odds, p / (1 − p) | 0.578 | 0.216 | Odds ratio 0.374: 63% lower odds |

The logit coefficient on member is the difference in log odds: ln 0.216 − ln 0.578 = −1.532 − (−0.548) = −0.984.

A 63% drop in the odds corresponds here to a 51% drop in the probability. Software and LLMs often report odds ratios, `np.exp(coef)`. If you meet
one, translate it into probabilities. In this course we read a coefficient's sign and significance, and report
probabilities and percentage points.

In [ ]:
shares = customers.groupby("member")["churned"].mean()
odds = shares / (1 - shares)
print(f"Odds, non-members: {odds[0]:.3f}")
print(f"Odds, members:     {odds[1]:.3f}")
print(f"Odds ratio:        {odds[1] / odds[0]:.3f}")
print(f"exp(coefficient):  {np.exp(m_one.params['member']):.3f}")

### Exercise 1. Channel in place of membership

**Predict.** Will a logit on channel alone give back the shares of app (23.8%) and web (36.5%) customers who
churned?

**Run** the cell. It fits the logit in `formula` and turns the coefficients into probabilities.

In [ ]:
formula = "churned ~ C(channel)"

m_ex1 = smf.logit(formula, data=customers).fit(disp=0)
print(m_ex1.params.round(4))

b = m_ex1.params
for channel in ["App", "Web"]:
    z = b["Intercept"]
    if channel == "Web":
        z = z + b["C(channel)[T.Web]"]
    p = 1 / (1 + np.exp(-z))
    print(f"{channel}: z = {z:.4f}, p = {p:.3f}")

print(customers.groupby("channel")["churned"].agg(["sum", "count", "mean"]).round(3))

**By hand.** Turn the two coefficients into probabilities with 1 / (1 + e<sup>−z</sup>) and compare with the cell.

**Answer.** Is the coefficient on Web a difference in probability? If not, what is it?

## Part 2. Fitting the model and reading the output

### The full model in Python

New today: `smf.logit` in place of `smf.ols`. The formula is the same as for the LPM in Lecture 8. The message
"Optimization terminated successfully" comes from `.fit()`: the coefficients are found by a search, step by step,
until it cannot improve. The function value it prints, 0.537380, is the log-likelihood divided by n with the
sign flipped (537.38 / 1,000).

In [ ]:
m_logit = smf.logit("churned ~ member + C(channel) + tenure_years + age + visits_2025", data=customers).fit()
print(m_logit.summary())

### Reading the coefficient table

- **The signs.** Web customers churn more often than app users; members, longer tenure and more visits go with less
  churn, each at the same values of the other x's.
- **z and P>|z|.** The test of H<sub>0</sub>: β = 0 as before. It says z because the standard errors rest on large
  samples.
- **Age.** No clear association once the others are in the model (p = 0.23).
- **What the size is not.** 0.4932 on Web is not 49 points, and not 0.49 points. It is the change in z, the log odds,
  and how much that moves the probability depends on the customer. Wooldridge 17-1d: the coefficients give the sign of
  each association and its significance. Sizes are read as probabilities in Parts 3 and 4.

### How the coefficients are found: maximum likelihood

Maximum likelihood chooses the values that make the data we observed as probable as possible (Wooldridge 17-1b and
C-4b). The cell below shows the idea for the simplest case. Among the non-members, 218 of 595 churned. For each
possible churn probability p, it computes how probable that outcome is, as a log (the log-likelihood). The
log-likelihood is highest at p = 0.366, the share.

For the full model, statsmodels searches over six coefficients, so that the predicted probabilities fit the observed
0s and 1s as well as possible. There is no formula as in least squares. The log-likelihood is negative (17-1c); closer to 0
is a better fit.

In [ ]:
churned_n, n = 218, 595          # non-members: churned and in total
p_values = np.arange(0.05, 0.9501, 0.001)
log_lik = []
for p in p_values:
    log_lik.append(churned_n * np.log(p) + (n - churned_n) * np.log(1 - p))

best = p_values[np.argmax(log_lik)]
print(f"Highest log-likelihood on the grid at p = {best:.3f}")
print(f"The share of non-members who churned: {churned_n / n:.3f}")
print(f"Log-likelihood at the share: {churned_n * np.log(churned_n / n) + (n - churned_n) * np.log(1 - churned_n / n):.2f}")

plt.figure(figsize=(7, 4))
plt.plot(p_values, log_lik)
plt.axvline(churned_n / n, color="darkorange", linestyle="--")
plt.xlabel("Churn probability p tried for the non-members")
plt.ylabel("Log-likelihood")
plt.show()

### Reading the top of the output

| In the output | What it tells you |
|---|---|
| Method: MLE; converged: True | Maximum likelihood, and the search found the top. If False, do not use the results. |
| Log-Likelihood −537.38 | The fit of this model |
| LL-Null −602.15 | The fit with only an intercept: every customer gets 29.0% |
| Pseudo R-squ. 0.1076 | McFadden's pseudo R², 1 − (−537.38 / −602.15) = 0.108 |
| LLR p-value 2.975e-26 | The test that all five slopes are 0, like the F test in OLS output |

The pseudo R² is 0 if the x's add nothing, and it can never reach 1. It is not a share of variance explained. Use it
to compare models for the same y on the same customers.

In [ ]:
print(f"Log-likelihood: {m_logit.llf:.2f}")
print(f"LL-Null:        {m_logit.llnull:.2f}")
print(f"Pseudo R2:      {m_logit.prsquared:.4f}")
print(f"By hand:        {1 - m_logit.llf / m_logit.llnull:.4f}")
print(f"Converged:      {m_logit.mle_retvals['converged']}")

### Comparing two models: the likelihood ratio test

LR = 2 (ℓ<sub>ur</sub> − ℓ<sub>r</sub>), read "two times ell-u-r minus ell-r", compared with a χ² distribution with q
degrees of freedom (Wooldridge 17-1c).

- ℓ<sub>ur</sub>: the log-likelihood of the larger model, without the restriction
- ℓ<sub>r</sub>: the log-likelihood of the smaller model, with q x's left out
- Both models must be fitted on the same customers.

A large LR means the extra x's raise the log-likelihood more than chance would. The test is useful for a group of x's,
such as the four region dummies (workshop). For one x, the z test in the table and the LR test usually agree.

**Example.** All five x's against the intercept only: LR = 2 × (−537.38 − (−602.15)) = 129.5 with 5 degrees of
freedom, p = 3.0 × 10<sup>−26</sup>. This is the LLR p-value in the output.

In [ ]:
lr = 2 * (m_logit.llf - m_logit.llnull)
print(f"LR = {lr:.1f}, p = {stats.chi2.sf(lr, 5):.1e}")

### Exercise 2. Does log spend help?

**Predict.** Will the log-likelihood go up, down or stay the same when ln spend in 2025 is added to the model?

**Run** the cell. It fits the model with and without the variable in `extra`.

In [ ]:
extra = "log_spend"

base = "churned ~ member + C(channel) + tenure_years + age + visits_2025"
m_without = smf.logit(base, data=customers).fit(disp=0)
m_with = smf.logit(base + " + " + extra, data=customers).fit(disp=0)

for name, m_ in [("without", m_without), ("with", m_with)]:
    print(f"{name:8s} log-likelihood {m_.llf:.2f}, pseudo R2 {m_.prsquared:.4f}")
print(f"{extra}: coef {m_with.params[extra]:.3f}, z {m_with.tvalues[extra]:.2f}, p {m_with.pvalues[extra]:.2f}")

**Compute.** LR = 2 × (ℓ with − ℓ without), and compare with 3.84, the 5% critical value of χ² with 1 degree of
freedom.

**Answer.** Does log spend help to predict churn, given the other x's? Run the next cell to check your LR.

In [ ]:
lr = 2 * (m_with.llf - m_without.llf)
print(f"LR = {lr:.2f}, p = {stats.chi2.sf(lr, 1):.2f}, critical value {stats.chi2.ppf(0.95, 1):.2f}")

## Part 3. Predicted probabilities for customers

**Case question.** How likely is a new app customer to stop buying, and how much lower is that probability for a
similar club member?

**What we know.** Customer A uses the app, has been a customer for 1 year, is aged 45 and had 15 visits in 2025, and
is not a member. We have the fitted logit model from Part 2.

**What we want to find out.** The predicted probability that customer A churns in the first half of 2026; the same
for customer A as a member, and the gap in percentage points. A predicted probability is the model's answer for a
customer we describe (Wooldridge 17-1d).

**By hand**, with the coefficients rounded to four decimals:

1. The index: z = 1.0309 − 0.1564 × 1 − 0.0083 × 45 − 0.0389 × 15 = −0.08 (app user, so Web = 0; not a member)
2. The probability: p = 1 / (1 + e<sup>0.08</sup>) = 0.48
3. As a member: z = −0.08 − 0.61 = −0.69, so p = 1 / (1 + e<sup>0.69</sup>) = 0.33

About 15 points lower as a member, for this customer. Rounding the coefficients moves the last digit; Python gives the
exact values.

**In Python.** Every variable in the formula needs a column; channel is given as text, as in the data. Two rows: the
same customer as a non-member and as a member.

In [ ]:
customer_a = pd.DataFrame({"member": [0, 1], "channel": "App",
                           "tenure_years": 1, "age": 45,
                           "visits_2025": 15})
print(m_logit.predict(customer_a).round(4))

**For the manager.** A new app customer with 15 visits in a year has a predicted probability of about 48% of
stopping in the first half of 2026. For a similar customer who is a club member it is about 34%, 14.5 percentage
points lower. Customers choose to join the club, so this compares customers as they are.

### The same membership, a different gap

Customer B also uses the app and is aged 45, but has been a customer for 8 years and had 60 visits.

In [ ]:
customer_b = pd.DataFrame({"member": [0, 1], "channel": "App",
                           "tenure_years": 8, "age": 45,
                           "visits_2025": 60})

for name, cust in [("A", customer_a), ("B", customer_b)]:
    p = m_logit.predict(cust)
    lpm = m_lpm.predict(cust)
    print(f"Customer {name}: logit {p.iloc[0]:.3f} and {p.iloc[1]:.3f}, gap {100 * (p.iloc[0] - p.iloc[1]):.1f} points;"
          f" LPM {lpm.iloc[0]:.3f} and {lpm.iloc[1]:.3f}")

In the LPM the gap is the coefficient, 10.7 points, for every customer, and for B it predicts probabilities below 0.
In the logit model the coefficient −0.605 moves z by the same amount for A and B, but the curve is steep where A is
and flat where B is. So in a logit model there is no single effect of membership on the probability; it depends on the
customer.

### Why the gap differs

The figure shows predicted churn against visits for app users with 3 years as customers, aged 45. The logit curves
come together as churn approaches 0, so the gap shrinks with more visits. The LPM lines keep the same distance and
cross 0 at about 62 visits for members and 80 for non-members. 35 of the 1,000 customers have more than 60 visits.
Long tenure works the same way: it also lowers z.

In [ ]:
visits = np.arange(0, 91, 2)
plt.figure(figsize=(8, 4.5))
for member, color in [(0, "tab:blue"), (1, "tab:orange")]:
    grid = pd.DataFrame({"member": member, "channel": "App", "tenure_years": 3, "age": 45, "visits_2025": visits})
    label = "member" if member == 1 else "not a member"
    plt.plot(visits, m_logit.predict(grid), color=color, label=f"logit, {label}")
    plt.plot(visits, m_lpm.predict(grid), color=color, linestyle="--", label=f"LPM, {label}")
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("Visits in 2025")
plt.ylabel("Predicted probability of churn")
plt.legend()
plt.show()

print(f"Customers with more than 60 visits: {(customers['visits_2025'] > 60).sum()}")

### Exercise 3. Customer A with more visits

**Predict.** Customer A gets 40 visits in place of 15. Will the membership gap grow or shrink?

**Run** the cell with `visits = 15` first, then **change** it to 40 and run it again.

In [ ]:
visits = 15     # change to 40

customer = pd.DataFrame({"member": [0, 1], "channel": "App", "tenure_years": 1, "age": 45,
                         "visits_2025": visits})
p = m_logit.predict(customer)
print(f"Not a member: {p.iloc[0]:.3f}")
print(f"Member:       {p.iloc[1]:.3f}")
print(f"Gap: {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")

**Compare** the gap with 14.5 points for 15 visits.

**Answer.** Use the curve: why does the gap change, although the coefficient is the same?

## Part 4. Averages and the choice of model

**Case question.** How much lower is the probability of churn for members than for non-members, on average over our
customers, at the same channel, tenure, age and visits?

**What we know.** Each customer has their own gap (Part 3). The LPM gave one number for all: 10.7 percentage points.

**What we want to find out.** One number that summarizes the gaps, in percentage points, with its standard error and
95% interval. Wooldridge calls it the average partial effect (APE) or, sometimes, the average marginal effect (AME)
(17-1d); statsmodels calls it the marginal effect, dy/dx.

### The average marginal effect by hand

AME = (1/n) Σ [p̂<sub>i</sub>(member = 1) − p̂<sub>i</sub>(member = 0)], read "one over n times the sum of p-hat-i with
member equal to one minus p-hat-i with member equal to zero".

1. **Predict twice:** for every customer, with member = 1 and with member = 0, all else as it is.
2. **Subtract:** that customer's gap.
3. **Average** over the 1,000 customers.

For a number such as visits: the slope of the curve for each customer (about the change for one more visit),
averaged in the same way.

In [ ]:
as_member = customers.copy()
as_member["member"] = 1
as_non_member = customers.copy()
as_non_member["member"] = 0

gaps = m_logit.predict(as_member) - m_logit.predict(as_non_member)
print(f"Gaps from {100 * gaps.max():.1f} to {100 * gaps.min():.1f} points")
print(f"Average: {100 * gaps.mean():.1f} points")

plt.figure(figsize=(7, 3.5))
plt.hist(gaps, bins=np.arange(-0.16, -0.0099, 0.01), edgecolor="white")
plt.xlabel("Gap in probability, member against not a member")
plt.ylabel("Customers")
plt.show()

### The average marginal effect in Python

`at="overall"` averages over the customers; `dummy=True` gives the change from 0 to 1 for a 0/1 variable. dy/dx is in
probability; times 100 it is percentage points.

In [ ]:
ame = m_logit.get_margeff(at="overall", dummy=True)
print(ame.summary())

**For the manager.** Comparing customers with the same channel, tenure, age and visits, members were on average 10.8
percentage points less likely to stop buying in the first half of 2026 (95% interval 5.1 to 16.5 points). Web
customers were 9.1 points more likely to stop than app users.

### Logit next to the LPM

On average the two models agree closely. Only the AMEs compare with the LPM; the logit coefficients are on another
scale. The models differ for single customers (Part 3) and in the predictions: the LPM has 41 below 0, the logit runs
from 0.017 to 0.701. Wooldridge 17-1d finds the same in his example.

In [ ]:
table = pd.DataFrame({"LPM coef, points": 100 * m_lpm.params.drop("Intercept"),
                      "logit AME, points": 100 * ame.margeff,
                      "logit coef": m_logit.params.drop("Intercept")})
print(table.round({"LPM coef, points": 2, "logit AME, points": 2, "logit coef": 4}))
print(f"Logit predictions from {m_logit.predict(customers).min():.3f} to {m_logit.predict(customers).max():.3f}")

### Three kinds of marginal effect, in Python and in Wooldridge

| Choice | Wooldridge 17-1d | statsmodels `get_margeff` | Here: member, Web (points) |
|---|---|---|---|
| Effect of a 0/1 variable | The change in probability from 0 to 1 | A derivative unless `dummy=True` | −10.92, 8.90 (derivative); −10.84, 9.12 (0 to 1) |
| Average over customers | Average partial effect (APE): each customer's effect, then the average | `at="overall"`, the default | −10.84, 9.12 |
| At the average customer | Partial effect at the average (PEA): put the averages into the model | `at="mean"` | −11.19, 9.59 |

Wooldridge prefers the change from 0 to 1 for a 0/1 variable, and points out that the average of a 0/1 variable (0.405
for member) describes no customer. We use `get_margeff(at="overall", dummy=True)`. Here the choices differ by less
than a point; near 0 or 1 they can differ more.

In [ ]:
for at, dummy in [("overall", False), ("overall", True), ("mean", True)]:
    me = m_logit.get_margeff(at=at, dummy=dummy).summary_frame()["dy/dx"]
    print(f"at={at}, dummy={dummy}: member {100 * me['member']:.2f}, Web {100 * me['C(channel)[T.Web]']:.2f}")

### How well does the model sort customers?

`pred_table()` counts how many customers the model puts on the right side of a threshold, 0.5 unless you choose
another. Wooldridge 17-1d: report how well each outcome is predicted, as well as the overall percent.

At 0.5 the model predicts 73.6% correctly; predicting that nobody churns gives 71.0%. It finds 72 of the 290
churners, 24.8%. With the threshold at the churn share, 0.29, it flags 447 customers and finds 193 of the 290 churners
(66.6%), with 64.2% of the stayers correct. The threshold is a business choice: how many customers can get a retention
offer, and what a missed churner costs.

In [ ]:
for threshold in [0.5, 0.29]:
    t = m_logit.pred_table(threshold=threshold)      # rows: stayed, churned; columns: predicted stays, churns
    print(f"Threshold {threshold}")
    print(t)
    print(f"  correct {(t[0, 0] + t[1, 1]) / t.sum():.3f}, churners found {t[1, 1]:.0f} of {t[1].sum():.0f},"
          f" flagged {t[:, 1].sum():.0f}")

### Probit: the normal curve

p<sub>i</sub> = Φ(z<sub>i</sub>) = P(Z ≤ z<sub>i</sub>), read "capital phi of z-i": the probability that a standard
normal Z is at most z<sub>i</sub>, as in the z table of Lectures 1 to 4. The index z<sub>i</sub> is the same as in
the logit model. The normal curve replaces the logistic curve; everything else is the same, and the output,
`predict` and `get_margeff` work as for logit. Wooldridge 17-1a: the two curves have almost the same shape.

### Exercise 4. Probit in place of logit

**Predict.** Which of these change when `smf.probit` replaces `smf.logit`: the coefficients, the AMEs, the predictions
for customers A and B?

**Run** the cell. It fits the model with `smf.logit` and with `second = smf.probit`, and prints the two side by side.

In [ ]:
second = smf.probit

formula = "churned ~ member + C(channel) + tenure_years + age + visits_2025"
models = {"logit": smf.logit(formula, data=customers).fit(disp=0),
          "probit": second(formula, data=customers).fit(disp=0)}

rows = {}
for name, m_ in models.items():
    me = m_.get_margeff(at="overall", dummy=True).summary_frame()["dy/dx"]
    pa, pb = m_.predict(customer_a), m_.predict(customer_b)
    rows[name] = {"coef member": m_.params["member"],
                  "coef visits_2025": m_.params["visits_2025"],
                  "AME member, points": 100 * me["member"],
                  "AME Web, points": 100 * me["C(channel)[T.Web]"],
                  "A, not a member": pa.iloc[0], "A, member": pa.iloc[1],
                  "B, not a member": pb.iloc[0], "B, member": pb.iloc[1],
                  "log-likelihood": m_.llf}
print(pd.DataFrame(rows).round(4))

**Compare** the two columns, row by row.

**Answer.** Why can the coefficients differ while the AMEs hardly do?

### Logit or probit

The two curves are nearly the same in the middle; in the tails logit approaches 0 and 1 more slowly. For customer B,
logit gives 5.1% and probit 4.1%. The log-likelihoods are −537.38 and −537.19, both with six coefficients; a
difference this small does not decide. Each model measures its coefficients on its own scale: here the logit coefficients
are about 1.7 times the probit ones (Wooldridge 17-1d gives 1.6 as a rule of thumb). Wooldridge gives no rule
for choosing. Probit is more common in econometrics, where a normal error is the usual assumption (17-1a); logit is
common elsewhere, partly because its coefficients link to odds. Report
one, say why, and check that the other gives the same conclusions.

### Choosing the LPM, logit or probit

| What you need from the model | Model |
|---|---|
| An average difference in percentage points, with few fitted values outside 0 to 1 | The LPM with robust standard errors is fine. Check it against the logit AME. |
| Probabilities for customers you describe, or many fitted values near or outside 0 and 1 | Logit or probit |
| Logit or probit? | Either. Say which and check with the other. |

In our case: the average gap for members is −10.7 points (LPM), −10.8 (logit) and −10.8 (probit), so any of the three
answers that question. For customer B as a member, the LPM predicts −11.7%, logit 2.8% and probit 1.8%, so for a
question about one customer, logit or probit.

## Reporting a logit model

We fitted a logit model for churn in the first half of 2026 on club membership, main channel, years as a customer,
age and visits in 2025, for a random sample of 1,000 customers (maximum likelihood; log-likelihood −537.38, pseudo R²
0.108). Comparing customers with the same channel, tenure, age and visits, members were on average 10.8 percentage
points less likely to churn (average marginal effect, 95% interval 5.1 to 16.5); a probit model and the LPM give nearly the
same average (10.8 and 10.7 points). The gap depends on the customer: for a new app customer with 15 visits the predicted probability is 48% as a
non-member and 34% as a member, for a customer with 8 years and 60 visits 5% and 3%. Customers choose to join the
club, so these are differences between customers as they are.

Report: the model, how it was fitted, and the log-likelihood; average marginal effects in percentage points, with
intervals; predicted probabilities for a few customers the reader cares about; association language, and odds ratios
only if asked for.

## Reference

### The steps for a yes/no outcome

| Step | What you do | In Python |
|---|---|---|
| Set up | y is 0 or 1; choose the x's; LPM, logit or probit | |
| Fit | Estimate the model by maximum likelihood | `smf.logit(formula, data=...).fit()` |
| Read | Signs, z and p; converged; log-likelihood, pseudo R², LLR p-value | `m.summary()` |
| Compare models | Likelihood ratio test for a group of x's | `2 * (m1.llf - m0.llf)`, `stats.chi2.sf` |
| Predict | Probabilities for customers you describe | `m.predict(new)` |
| Average | AMEs in percentage points | `m.get_margeff(at="overall", dummy=True)` |
| Check the choice | Probit and the LPM: do the conclusions hold? | `smf.probit`, `smf.ols(...).fit(cov_type="HC1")` |
| Report | Probabilities and percentage points, as associations | |

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| z<sub>i</sub> | z-i | The index for customer i: β<sub>0</sub> + β<sub>1</sub> x<sub>1i</sub> + ..., any number |
| p<sub>i</sub> | p-i | The predicted probability that customer i has y = 1 |
| 1 / (1 + e<sup>−z</sup>) | one over one plus e to the minus z | The logistic curve (logit); Wooldridge's Λ(z) |
| Φ(z) | capital phi of z | The standard normal curve, P(Z ≤ z) (probit) |
| p / (1 − p) | p over one minus p | The odds |
| ℓ | ell | The log-likelihood |
| ℓ<sub>ur</sub>, ℓ<sub>r</sub> | ell-u-r, ell-r | The log-likelihoods of the larger and the smaller model |
| LR | L-R | The likelihood ratio statistic, 2 (ℓ<sub>ur</sub> − ℓ<sub>r</sub>) |
| AME | A-M-E | The average marginal effect: each customer's change in probability, averaged |

### Danish terms

| English | Danish |
|---|---|
| yes/no outcome | binært udfald |
| predicted probability | forudsagt sandsynlighed |
| logit model | logitmodel |
| probit model | probitmodel |
| odds ratio | oddsratio |
| maximum likelihood estimation | maksimum likelihood-estimation |
| likelihood ratio test | likelihood ratio-test, kvotienttest |
| standard normal distribution | standardnormalfordeling |